# `make format` and `make check`

You run two commands around every change to QMCPy:

| command | when | what it does |
|---|---|---|
| **`make format`** | before you commit | **changes your files**: imports, whitespace, asserts, docstring types |
| **`make check`** | before you open a PR | **only reads**: the same rules CI enforces |

Each command runs a few small tools in order. This notebook walks through both, shows what each tool does on a tiny example, then lists the rest.

The code cells run the real scripts when you are inside a QMCSoftware checkout. From a plain `pip install qmcpy` they print the same before/after as text.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMCSoftware/blob/develop/demos/makefile_dev_tools.ipynb)

In [1]:
# @title Execute this cell to install dependencies
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False
if IN_COLAB:
  !pip install -q qmcpy

In [2]:
# Every import the notebook uses, in one place.
import json, os, pathlib, re, shutil, subprocess, sys, tempfile

# The QMCSoftware checkout, or None when this notebook runs on its own.
REPO = next((d for d in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
             if (d / "scripts/convert_asserts.py").exists()), None)

def demo(tool, before, after, target, name="snippet.py"):
    """Print `Before`, then `After` running `tool` on it.

    `tool` is the argument list for a script under scripts/. Inside a checkout
    the real script runs and its result replaces `after`; otherwise the canned
    `after` text is used.
    """
    if REPO:
        f = pathlib.Path(tempfile.mkdtemp()) / name
        f.write_text(before)
        try:
            subprocess.run([sys.executable, *tool, str(f)], cwd=REPO,
                           capture_output=True, text=True, check=True)
            after = f.read_text()
        except (FileNotFoundError, subprocess.CalledProcessError):
            pass
    print("Before:\n-------\n" + before)
    after_str = f"After (make {target}):"
    print(f"{after_str}\n" + "-"*len(after_str) + "\n" + after)

## 1. The two pipelines

`make format` and `make check` are just ordered lists of smaller targets. Here they are, read straight from the `makefile`:

In [3]:
DEFAULT = {
    "format": "flatten_qmcpy_imports markdown-unwrap rm_trailing_whitespace "
              "harden_colab_notebook convert_asserts_changed add_docstring_arg_types_changed".split(),
    "check": "check_test_style check_docstring_changed check_baseline "
             "check_asserts_changed check_links".split(),
}

def steps(target):
    """The sub-targets `make <target>` runs, read from the makefile."""
    makefile = REPO / "makefile" if REPO else None
    if makefile and makefile.exists():
        block = re.search(rf"^{target}:(?:.*\n)((?:[ \t].*\n|\n)+)", makefile.read_text(), re.M)
        names = re.findall(r"\$\(MAKE\)\s+(\S+)", block.group(1)) if block else []
        if names:
            return names
    return DEFAULT[target]

for target in ("format", "check"):
    print(f"make {target}")
    for name in steps(target):
        print("  ", name)

make format
   flatten_qmcpy_imports
   markdown-unwrap
   rm_trailing_whitespace
   harden_colab_notebook
   convert_asserts_changed
   add_docstring_arg_types_changed
make check
   check_test_style
   check_docstring_changed
   check_baseline
   check_asserts_changed
   check_links


Every step ends with **one summary line**, so a run is easy to scan:

- `clean  (0/42 files)` — nothing to do
- `3 changed  (3/42 files)` — `make format` rewrote 3 files
- `WARNING: 2 problem(s)  (2/42 files)` — issues found, but this step does not fail the build
- `ERROR: 2 problem(s)  (2/42 files)` — issues found and this step fails (same as CI)

Any details are listed just above that line, one `-` bullet each.

## 2. `make format`: changes your files

Run it before you commit. Every step is safe to run again and touches only what it needs to.

| step | what it does |
|---|---|
| `flatten_qmcpy_imports` | rewrite `from qmcpy.sub.mod import X` as `import qmcpy as qp` then `qp.X` |
| `markdown-unwrap` | join hard-wrapped Markdown lines back into one line per paragraph |
| `rm_trailing_whitespace` | remove trailing spaces across the repo |
| `harden_colab_notebook` | add the *Open in Colab* badge to any demo notebook that lacks one |
| `convert_asserts_changed` | turn `assert` into a real `raise`, in changed files (next cell) |
| `add_docstring_arg_types_changed` | copy signature types into `Args:` lines, in changed files (cell after) |

`format` has no docstring reformatter. `format-docstring` was tried and dropped: on this code it deletes `Returns:` and `Yields:` types and turns `**References:**` into `**References: **`.

### 2.1 `flatten_qmcpy_imports`

Collapse a deep import path to the one public name. Anything exported by `qmcpy` should be reached as `qp.<Name>`, so imports stay stable when internal modules move.

In [4]:
demo(["scripts/flatten_qmcpy_imports.py"],
"""from qmcpy.discrete_distribution.lattice.lattice import Lattice
""",
"""from qmcpy import Lattice
""", "flatten_qmcpy_imports")

Before:
-------
from qmcpy.discrete_distribution.lattice.lattice import Lattice

After (make flatten_qmcpy_imports):
-----------------------------------
from qmcpy import Lattice



### 2.2 `markdown_unwrap`

Join each hard-wrapped Markdown paragraph back onto one line (code fences and math are left alone), so a later edit shows as a word change, not a whole-paragraph rewrap.

In [5]:
demo(["scripts/unwrap_markdown.py"],
"QMCPy estimates an integral as the mean of an\n"
"integrand sampled on a low-discrepancy point\n"
"set, and stops once the error is small enough.\n",
"QMCPy estimates an integral as the mean of an integrand sampled on a "
"low-discrepancy point set, and stops once the error is small enough.\n",
"markdown_unwrap", name="snippet.md")

Before:
-------
QMCPy estimates an integral as the mean of an
integrand sampled on a low-discrepancy point
set, and stops once the error is small enough.

After (make markdown_unwrap):
-----------------------------
QMCPy estimates an integral as the mean of an integrand sampled on a low-discrepancy point set, and stops once the error is small enough.



### 2.3 `rm_trailing_whitespace`

Strip spaces and tabs at end of line across every git-tracked text file. Nothing to run on a snippet here — it asks git which files exist. Effect: `"muhat = 1.80   \n"` becomes `"muhat = 1.80\n"`.

### 2.4 `harden_colab_notebook`

Add the *Open in Colab* badge cell and the dependency-install cell to any notebook under `demos/` that is listed in the Colab manifest but missing them. It works on real notebook files, not snippets; `make format` runs it for every still-unclassified notebook.

### 2.5 `convert_asserts`

`python -O` removes every `assert`, so a check written that way is gone when Python runs with `-O`. This rewrites it as a real `raise`, keeping comments and layout.

In [6]:
demo(["scripts/convert_asserts.py", "--exception", "AssertionError"],
"""def clip(x, lo, hi):
    assert lo <= hi, "empty interval"
""",
"""def clip(x, lo, hi):
    if not (lo <= hi):
        raise AssertionError("empty interval")
""", "convert_asserts_changed")

Before:
-------
def clip(x, lo, hi):
    assert lo <= hi, "empty interval"

After (make convert_asserts_changed):
-------------------------------------
def clip(x, lo, hi):
    if not (lo <= hi):
        raise AssertionError("empty interval")



### 2.6 `add_docstring_arg_types`

The type is already in the signature. This copies it into the matching `Args:` line (`name` becomes `name (type)`). It never makes up a type or a description.

In [7]:
src = """def disc_area(radius: float, n_sectors: int = 4) -> float:
    \"\"\"Area of a disc.

    Args:
        radius: distance to the edge.
        n_sectors: wedge count.
    \"\"\"
    return 3.14159 * radius ** 2
"""

demo(["scripts/add_docstring_arg_types.py"], src,
     src.replace("radius:", "radius (float):").replace("n_sectors:", "n_sectors (int):"),
     "add_docstring_arg_types_changed")

Before:
-------
def disc_area(radius: float, n_sectors: int = 4) -> float:
    """Area of a disc.

    Args:
        radius: distance to the edge.
        n_sectors: wedge count.
    """
    return 3.14159 * radius ** 2

After (make add_docstring_arg_types_changed):
---------------------------------------------
def disc_area(radius: float, n_sectors: int = 4) -> float:
    """Area of a disc.

    Args:
        radius (float): distance to the edge.
        n_sectors (int): wedge count.
    """
    return 3.14159 * radius ** 2



## 3. `make check`: only reads

Run it before you open a PR. Nothing here changes your files. It runs the same checks as the *Check test-suite conventions* job in CI.

| step | what it checks |
|---|---|
| `check_test_style` | each `test/` file is named `test_<area>_*.py` and uses a `unittest.TestCase` class (next cell) |
| `check_docstring_changed` | Google-style format plus `pydoclint`, on your changed files |
| `check_baseline` | the count of known issues did not go up (cell after) |
| `check_asserts_changed` | a dry run of `convert_asserts` on changed source |
| `check_links` | internal doc links and anchors resolve |

Left out on purpose: `check_links_external` (needs the network) and `check_pep8_changed` (too many old violations to be a useful gate today).

### 3.1 `check_test_style`

One file that follows both rules, one that breaks both:

In [8]:
folder = pathlib.Path(tempfile.mkdtemp())
(folder / "test_tm_ok.py").write_text(
    "import unittest\nclass T(unittest.TestCase):\n    def test_x(self): self.assertEqual(2, 2)\n")
(folder / "test_zz_bad.py").write_text("def test_x():\n    assert 2 == 2\n")

if REPO:
    # run from inside `folder` so the report shows plain file names
    result = subprocess.run([sys.executable, str(REPO / "scripts/check_test_style.py"), "."],
                            cwd=folder, capture_output=True, text=True)
    print(result.stdout)
else:
    print("test_zz_bad.py is flagged twice: it uses a bare function, and 'zz' is not a known area.")


  - 1 of 2 files use a unittest.TestCase class
  - 1 file(s) use bare pytest functions (no unittest.TestCase class):
    - test_zz_bad.py
  - 1 of 2 files use a test_<area>_ prefix (dd, ee, ft, ig, kn, sc, sr, tm, ut)
  - 1 file(s) have no recognized test_<area>_ prefix:
    - test_zz_bad.py



### 3.2 `check_docstring`

Two passes over public docstrings under `qmcpy/`: `check_docstring.py` for Google-style **format** (a one-line summary first, a blank line before each `Args:` / `Returns:`, canonical `Name:` headers) and `pydoclint` for **content** (every parameter and return documented). Informational unless `STRICT=--strict`.

In [9]:
mod = pathlib.Path(tempfile.mkdtemp()) / "mod.py"
mod.write_text(
    'def disc_area(radius):\n'
    '    """\n'
    '    Args:\n'
    '        radius (float): distance to the edge.\n'
    '    """\n'
    '    return 3.14159 * radius ** 2\n')

if REPO:
    print(subprocess.run([sys.executable, str(REPO / "scripts/check_docstring.py"), mod.name],
                         cwd=mod.parent, capture_output=True, text=True).stdout)
else:
    print("  - mod.py:3: missing-summary: docstring opens with `Args:`; add a one-line summary first")
    print("WARNING: 1 problem(s)  (1/1 files)")
print("With STRICT=--strict the last line becomes 'ERROR: ...' and the exit is 1.")


  - mod.py:3: missing-summary: docstring opens with `Args:`; add a one-line summary first
  - 1 file(s) scanned: 1 issue(s) across 1 file(s): 1 missing-summary

With STRICT=--strict the last line becomes 'ERROR: ...' and the exit is 1.


### 3.3 `check_baseline`

Some checks have many old violations, so they cannot fail the build yet. Their counts are stored in a file, and the build fails only if a count goes up.

In [10]:
path = REPO / "scripts/baseline_counts.json" if REPO else None
counts = (json.loads(path.read_text()) if path and path.exists()
          else {"check_docstring": 0, "pydoclint": 0, "unsafe_annotations": 0})
print("stored counts:", counts)
print("make check_baseline fails if any count goes above this.")
print("make check_baseline_update saves new counts after you fix or accept a change.")

stored counts: {'check_docstring': 0, 'pydoclint': 0, 'unsafe_annotations': 0}
make check_baseline fails if any count goes above this.
make check_baseline_update saves new counts after you fix or accept a change.


### 3.4 `check_asserts`

The read-only half of `convert_asserts` (2.5): same detection, but it writes nothing and exits non-zero when an `assert` in changed code could be converted. This is the variant `make check` runs.

### 3.5 `check_links`

Build the docs site (`mkdocs build`) and check that every internal link and heading anchor resolves. No inline example — it needs the built `site/`. A failure reads:

```
  - path/page.html: broken internal link '../missing.md'
ERROR: 3 problem(s)  (2/90 pages)
```

## 4. Other tools

Things `format` and `check` do not call directly:

| tool | what it is for |
|---|---|
| **`$(PYTHON)`** | the makefile finds the interpreter once (active env, then `$CONDA_PREFIX/bin/python`, then `conda run -n qmcpy`, then `python3`) and every rule uses it (next cell) |
| **the `_changed` suffix** | most tools have a whole-repo form and a `_changed` form that looks only at files changed from a base branch. On a large codebase, only the `_changed` form is practical day to day |
| **`annotate_public_api_types_changed`** and **`sync_docstring_types_changed`** | the reverse of `add_docstring_arg_types`: copy a documented type into a missing signature annotation, then copy annotations back into the docstrings. `check_public_api_types_changed` only reports (last cell) |
| **`tests_fast`** | run doctests, unit tests, and notebook tests at the same time; the rule records each exit code so a failing suite fails the target |
| **whole-repo forms** | `convert_asserts`, `add_docstring_arg_types`, `check_docstring` also exist without `_changed`, for a full pass |

In [11]:
def discover_python():
    """The makefile's  PYTHON ?= $(shell ...)  cascade, written in Python."""
    env = os.environ.get("CONDA_PREFIX")
    if shutil.which("python"):          # 1. active env on PATH
        return shutil.which("python")
    if env and shutil.which("python", path=env + "/bin"):   # 2. active env, not on PATH
        return shutil.which("python", path=env + "/bin")
    if shutil.which("conda"):                               # 3. the repo's qmcpy env
        out = subprocess.run(["conda", "run", "-n", "qmcpy", "python",
                              "-c", "import sys; print(sys.executable)"],
                             capture_output=True, text=True)
        if out.stdout.strip():
            return out.stdout.strip()
    return shutil.which("python3") or sys.executable        # 4. system python3


print("interpreter:", discover_python())

interpreter: /Users/terrya/miniconda3/envs/qmcpy/bin/python


### 4.1 `annotate_public_api_types`

The other direction: the types are only in the docstring and the signature is bare.

In [12]:
src = """def disc_area(radius, n_sectors=4):
    \"\"\"Area of a disc.

    Args:
        radius (float): distance to the edge.
        n_sectors (int): wedge count.

    Returns:
        float: the area.
    \"\"\"
    return 3.14159 * radius ** 2
"""

demo(["-m", "scripts.annotate_public_api_types"], src,
     src.replace("(radius, n_sectors=4)", "(radius: float, n_sectors: int = 4) -> float"),
     "annotate_public_api_types_changed")

Before:
-------
def disc_area(radius, n_sectors=4):
    """Area of a disc.

    Args:
        radius (float): distance to the edge.
        n_sectors (int): wedge count.

    Returns:
        float: the area.
    """
    return 3.14159 * radius ** 2

After (make annotate_public_api_types_changed):
-----------------------------------------------
def disc_area(radius: float, n_sectors: int = 4) -> float:
    """Area of a disc.

    Args:
        radius (float): distance to the edge.
        n_sectors (int): wedge count.

    Returns:
        float: the area.
    """
    return 3.14159 * radius ** 2



## 5. Takeaways

- Run `make format` before you commit, `make check` before you open a PR. One changes files, one only reads.
- Find the interpreter once. `$(PYTHON)` as a resolved variable removes a class of "wrong Python" bugs.
- Scope checks to the diff. A `_changed` target makes a whole-repo linter usable.
- Add a strict check to old code with a baseline count, not a hard gate.
- Use a codemod, not hand edits, for the assert and type changes, and keep a careless reformatter out of the pipeline.